In [1]:
import pandas as pd

df = pd.read_csv('../../data/novamarket_con_features.csv')

# S14 Seleccion de caracteristicas


## Pasos iniciales

In [2]:
df["fuga_cliente"].value_counts()

fuga_cliente
0    451579
1    119397
Name: count, dtype: int64

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
#Definicion de variables predictoras y variable objetivo
X = df.drop(columns=["fuga_cliente", "id_cliente"])
X = X.select_dtypes(include=["number", "bool"]) #Por un error en el filtro "ValueError: could not convert string to float: 'P668231'"
y = df["fuga_cliente"]


In [5]:
#Definicion de conjunto de entrenamiento y conjunto de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [6]:
#Verificacion
print("len(X_entrenamiento):", len(X_train))
print("len(X_prueba):", len(X_test))
print(len(X_train) + len(X_test) == len(X))

len(X_entrenamiento): 456780
len(X_prueba): 114196
True


Se confirma que la particion esta bien hecha

## Filtro

### Deteccion de varianza y redundancia

In [7]:
#Variance Treshold
from sklearn.feature_selection import VarianceThreshold

selector_varianza = VarianceThreshold(threshold=0.01)
selector_varianza.fit(X_train)
columnas_con_varianza = X_train.columns[selector_varianza.get_support()]

In [8]:
#Matriz de correlacion entre features con entrenamiento.corr()
# 1) Columnas que pasaron el filtro de varianza sin constantes
X_var = X_train[columnas_con_varianza].astype(float)

# 2) Matriz de correlacion entre features
matriz_corr = X_var.corr()

# 3) Pares con |r| > 0.85 (solo triangulo superior, sin repetir ni la diagonal)
umbral = 0.85
pares = []
cols = matriz_corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = matriz_corr.iloc[i, j]
        if abs(r) > umbral:
            pares.append((cols[i], cols[j], r))

pares_df = (pd.DataFrame(pares, columns=["feature_1", "feature_2", "correlacion"])
              .sort_values("correlacion", key=abs, ascending=False)
              .reset_index(drop=True))
pares_df

,feature_1,feature_2,correlacion
0,comentario_cliente_faltante,tiene_comentario,-1.000000
1,monto_compra,monto_compra_escalada,1.000000
2,precio_unitario,precio_unitario_escalada,1.000000
3,edad_cliente,edad_cliente_escalada,1.000000
4,unidades_vendidas,unidades_vendidas_escalada,1.000000
5,mes_pedido,trimestre_pedido,0.971638
6,longitud_comentario,num_palabras_comentario,0.947174
7,cat_Moda,stock_minimo,0.917781
8,dias_desde_ultima_compra,score_R,-0.901781
9,monto_compra_escalada,monto_compra_boxcox,0.877198


In [9]:
#lista de columnas descartadas por varianza
columnas_descartadas_varianza = [c for c in X_train.columns if c not in columnas_con_varianza]

print(f"Columnas descartadas por varianza ({len(columnas_descartadas_varianza)}):")
print(columnas_descartadas_varianza)


Columnas descartadas por varianza (4):
['precio_unitario_era_nulo', 'menciona_demora', 'dias_desde_ultima_compra_era_nulo', 'compras_ultimos_30_dias']


In [10]:
# Lista de pares de features redundantes por correlacion (|r| > 0.85)
print(f"Pares redundantes por correlacion ({len(pares_df)}):")
for _, fila in pares_df.iterrows():
    print(f"{fila['feature_1']}  <->  {fila['feature_2']}   r = {fila['correlacion']:.3f}")

Pares redundantes por correlacion (13):
comentario_cliente_faltante  <->  tiene_comentario   r = -1.000
monto_compra  <->  monto_compra_escalada   r = 1.000
precio_unitario  <->  precio_unitario_escalada   r = 1.000
edad_cliente  <->  edad_cliente_escalada   r = 1.000
unidades_vendidas  <->  unidades_vendidas_escalada   r = 1.000
mes_pedido  <->  trimestre_pedido   r = 0.972
longitud_comentario  <->  num_palabras_comentario   r = 0.947
cat_Moda  <->  stock_minimo   r = 0.918
dias_desde_ultima_compra  <->  score_R   r = -0.902
monto_compra_escalada  <->  monto_compra_boxcox   r = 0.877
monto_compra  <->  monto_compra_boxcox   r = 0.877
numero_compras  <->  score_F   r = 0.866
total_gastado  <->  score_M   r = 0.857


### Calculo de correlacion entre features con fuga_cliente

In [11]:
correlaciones_con_objetivo = X_train.corrwith(y_train).abs().sort_values(ascending=False)

correlaciones_con_objetivo.head(10) 

C:\Users\diego\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\diego\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


nivel_lealtad_cod           0.218324
monto_compra_boxcox         0.102953
monto_compra_log            0.084226
monto_compra                0.080829
monto_compra_escalada       0.080829
precio_unitario             0.058746
precio_unitario_escalada    0.058746
edad_cliente_escalada       0.048478
edad_cliente                0.048478
canal_App                   0.047786
dtype: float64

## Wrapper

### RFE con LogisticRegression


In [13]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Escalar (ajustado solo con entrenamiento) conservando nombres de columnas
escalador = StandardScaler().fit(X_train)
X_train_esc = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)

modelo_base = LogisticRegression(max_iter=1000)
selector_rfe = RFE(estimator=modelo_base, n_features_to_select=8)
selector_rfe.fit(X_train_esc, y_train)

features_elegidas_rfe = X_train.columns[selector_rfe.support_]
features_elegidas_rfe

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [14]:
#Ranking de las otras features
ranking_rfe = (
    pd.DataFrame({
        "feature": X_train.columns,
        "ranking": selector_rfe.ranking_,
        "seleccionada": selector_rfe.support_
    })
    .sort_values("ranking")
    .reset_index(drop=True)
)

ranking_rfe

AttributeError: 'RFE' object has no attribute 'ranking_'

## Embedded

### Entrenamiento con Random Forest, extraccion de importancias con tabla ordenada

In [15]:
from sklearn.ensemble import RandomForestClassifier

modelo_bosque = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_bosque.fit(X_train, y_train)

importancias = pd.Series(
    modelo_bosque.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

KeyboardInterrupt: 

In [ ]:
tabla_importancias = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importancia": modelo_bosque.feature_importances_
    })
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)

tabla_importancias

,feature,importancia
0,gasto_promedio,0.051072
1,total_gastado,0.049567
2,nivel_lealtad_cod,0.049409
3,dias_desde_ultima_compra,0.048190
4,peso_pedido_kg,0.047527
5,codigo_postal,0.045942
6,precio_unitario_escalada,0.044710
7,precio_unitario,0.044646
8,puntos_fidelidad_actuales,0.044214
9,monto_compra_boxcox,0.039249


## Comparacion de los 3 metodos

### Tabla comparativa


In [ ]:
candidatas = X_train.columns

tabla_metodos = pd.DataFrame({
    # Filtro: entre las 10 features con mayor correlacion absoluta con fuga_cliente
    "filtro": candidatas.isin(correlaciones_con_objetivo.dropna().head(10).index),

    # Envoltura: seleccionada por RFE
    "rfe": selector_rfe.support_,

    # Embebido: entre las 8 con mayor importancia del Random Forest
    "random_forest": candidatas.isin(tabla_importancias["feature"].head(8)),
}, index=candidatas)

# Tabla ordenada de las features elegidas por mas metodos
tabla_metodos["n_metodos"] = tabla_metodos.sum(axis=1)

pd.set_option("display.max_rows", None)
tabla_metodos.sort_values("n_metodos", ascending=False)


,filtro,rfe,random_forest,n_metodos
feature,,,,
nivel_lealtad_cod,True,True,True,3
precio_unitario,True,False,True,2
edad_cliente,True,True,False,2
precio_unitario_escalada,True,False,True,2
monto_compra_escalada,True,True,False,2
monto_compra,True,True,False,2
edad_cliente_escalada,True,True,False,2
monto_compra_log,True,True,False,2
monto_compra_boxcox,True,True,False,2


### Tabla organizada por cantidad de métodos

In [ ]:
cols_metodos = ["filtro", "rfe", "random_forest"]

# Si antes agregaste la columna "n_metodos", se elimina para no confundirla
tabla_metodos = tabla_metodos.drop(columns="n_metodos", errors="ignore")

tabla_metodos["veces_seleccionada"] = tabla_metodos[cols_metodos].sum(axis=1)

tabla_metodos = tabla_metodos.sort_values("veces_seleccionada", ascending=False)

pd.set_option("display.max_rows", None)
tabla_metodos

,filtro,rfe,random_forest,veces_seleccionada
feature,,,,
nivel_lealtad_cod,True,True,True,3
precio_unitario,True,False,True,2
edad_cliente,True,True,False,2
precio_unitario_escalada,True,False,True,2
monto_compra_escalada,True,True,False,2
monto_compra,True,True,False,2
edad_cliente_escalada,True,True,False,2
monto_compra_log,True,True,False,2
monto_compra_boxcox,True,True,False,2


### Features elegidas

| Feature | Métodos | Justificación de negocio |
|---|---|---|
| `nivel_lealtad_cod` | 3 | Resume qué tan comprometido está el cliente con NovaMarket. Un cliente de nivel bajo tiene menos incentivos para quedarse, por lo que es el candidato natural para campañas de retención y beneficios del programa de fidelidad. |
| `precio_unitario` | 2 | Indica el rango de precios en el que compra el cliente. Permite identificar si la fuga se concentra en clientes sensibles al precio, y orientar descuentos o promociones hacia ese segmento. |
| `edad_cliente` | 2 | Permite segmentar la base por etapa de vida. Con ella, marketing puede diseñar ofertas y canales de comunicación distintos para cada grupo etario. |
| `monto_compra_log` | 2 | Mide cuánto vale cada pedido para el negocio. Distingue a los clientes de ticket alto, cuya pérdida cuesta más, de los de ticket bajo. Se usa la versión logarítmica por reducir el efecto de pedidos extremos. |
| `canal_App` | 2 | Indica si el cliente compra por la app. Los clientes que usan canales propios son más fáciles de contactar y fidelizar, mientras que los que dependen de otros canales pueden estar más expuestos a la competencia. |
| `dias_desde_ultima_compra` | 1 | Mide el tiempo sin comprar. Es la señal de alerta temprana más usada en retención: un cliente que se demora más de lo habitual en volver tiene más riesgo de abandono, y permite activar campañas de reactivación a tiempo. |
| `total_gastado` | 1 | Mide cuánto dinero ha aportado el cliente en su historia. Permite priorizar el presupuesto de retención en los clientes más valiosos, donde cada fuga evitada tiene mayor retorno. |


## Carga de dataset final

In [ ]:
import os

lista_features_finales = [
    "nivel_lealtad_cod",
    "precio_unitario",
    "edad_cliente",
    "monto_compra_log",
    "canal_App",
    "dias_desde_ultima_compra",
    "total_gastado",
]

df_final = df[lista_features_finales + ["fuga_cliente", "id_cliente"]]

os.makedirs("Datos", exist_ok=True)
df_final.to_csv("Datos/novamarket_features_seleccionadas.csv", index=False)

# Verificacion: recargar y comprobar columnas
df_check = pd.read_csv("Datos/novamarket_features_seleccionadas.csv")
print(df_check.shape, df_final.shape, df_check.shape == df_final.shape)
print(list(df_check.columns))
print(set(df_check.columns) == set(lista_features_finales + ["fuga_cliente", "id_cliente"]))

(570976, 9) (570976, 9) True
['nivel_lealtad_cod', 'precio_unitario', 'edad_cliente', 'monto_compra_log', 'canal_App', 'dias_desde_ultima_compra', 'total_gastado', 'fuga_cliente', 'id_cliente']
True
